# 01. 바운딩 박스, IoU, 신뢰도 기초

## 학습 목표

YOLO v1을 이해하는 데 필요한 좌표 표현, 교집합/합집합, IoU, 신뢰도와 클래스별 점수를 NumPy로 직접 계산합니다.

## 실행 방법

저장소 루트에서 `python -m pip install jupyter numpy`를 실행한 뒤 `jupyter lab`로 이 파일을 엽니다. 위에서 아래로 셀을 실행하세요.

좌표 계약: 모든 박스는 이미지 크기로 나눈 0~1 범위이며, 중심 형식은 `[cx, cy, w, h]`, 모서리 형식은 `[x1, y1, x2, y2]`입니다.

In [ ]:
import numpy as np  # np라는 짧은 별칭으로 NumPy를 불러와 배열 계산을 명확하고 빠르게 수행합니다.

np.set_printoptions(precision=4, suppress=True)  # 출력 소수 자릿수를 4자리로 제한하고 작은 수의 지수 표기를 억제합니다.

## 1. 중심 좌표를 모서리 좌표로 바꾸기

`...`은 앞쪽 차원 개수와 무관하게 마지막 축을 선택하는 NumPy 문법입니다. 그래서 한 박스 `(4,)`와 여러 박스 `(N, 4)`를 같은 함수가 처리합니다.

In [ ]:
def center_to_corners(boxes):  # 함수 정의문 def는 반복해서 쓸 좌표 변환 절차에 이름을 붙입니다.
    boxes = np.asarray(boxes, dtype=np.float64)  # 입력을 64비트 실수 배열로 통일해 나눗셈과 제곱근의 정밀도를 확보합니다.
    if boxes.shape[-1] != 4:  # 마지막 축이 cx, cy, w, h 네 값인지 검사해 잘못된 입력을 조기에 막습니다.
        raise ValueError("boxes의 마지막 차원은 4여야 합니다.")  # raise는 조용히 잘못 계산하지 않고 즉시 원인을 알립니다.
    if np.any(boxes[..., 2:] < 0.0):  # 폭과 높이 중 음수가 하나라도 있는지 불리언 배열을 any로 축약합니다.
        raise ValueError("박스의 폭과 높이는 음수일 수 없습니다.")  # 기하학적으로 불가능한 박스를 거부합니다.

    center = boxes[..., :2]  # 슬라이스 :2는 마지막 축의 cx와 cy만 선택합니다.
    half_size = boxes[..., 2:] / 2.0  # 폭과 높이의 절반이 중심에서 각 모서리까지의 거리입니다.
    top_left = center - half_size  # 왼쪽 위 좌표는 중심에서 반 크기를 뺍니다.
    bottom_right = center + half_size  # 오른쪽 아래 좌표는 중심에 반 크기를 더합니다.
    return np.concatenate([top_left, bottom_right], axis=-1)  # 두 배열을 마지막 축으로 이어 [x1,y1,x2,y2]를 반환합니다.


sample_center_boxes = np.array(  # 두 박스를 한 번에 계산하려고 2차원 배열을 만듭니다.
    [[0.50, 0.50, 0.40, 0.20], [0.25, 0.30, 0.10, 0.30]],  # 각 행은 [중심 x, 중심 y, 폭, 높이]입니다.
    dtype=np.float64,  # 정수로 오해하지 않도록 자료형을 명시합니다.
)
sample_corner_boxes = center_to_corners(sample_center_boxes)  # 작성한 함수를 호출해 모서리 형식으로 변환합니다.
print(sample_corner_boxes)  # 예상 첫 행은 [0.3, 0.4, 0.7, 0.6]입니다.

assert np.allclose(sample_corner_boxes[0], [0.30, 0.40, 0.70, 0.60])  # 부동소수점 비교에는 오차를 허용하는 allclose를 씁니다.

## 2. IoU를 직접 구현하기

IoU(Intersection over Union)는 `교집합 넓이 / 합집합 넓이`입니다. 1이면 두 박스가 완전히 같고, 0이면 겹치지 않습니다. YOLO v1에서는 학습 시 어떤 예측기가 객체를 담당하는지 정하고 신뢰도 목표를 만들 때 중요합니다.

In [ ]:
def box_area(corner_boxes):  # 모서리 형식 박스의 넓이를 계산하는 보조 함수를 정의합니다.
    corner_boxes = np.asarray(corner_boxes, dtype=np.float64)  # 리스트와 배열 입력을 같은 형태로 정규화합니다.
    widths_heights = np.maximum(corner_boxes[..., 2:] - corner_boxes[..., :2], 0.0)  # 뒤-앞 좌표를 계산하고 음수 길이는 0으로 막습니다.
    return widths_heights[..., 0] * widths_heights[..., 1]  # 폭과 높이를 원소별로 곱해 넓이를 반환합니다.


def intersection_over_union(boxes_a, boxes_b, epsilon=1e-12):  # 같은 위치의 박스 쌍마다 IoU를 계산합니다.
    boxes_a = np.asarray(boxes_a, dtype=np.float64)  # 첫 번째 입력을 실수 배열로 바꿉니다.
    boxes_b = np.asarray(boxes_b, dtype=np.float64)  # 두 번째 입력도 같은 자료형으로 바꿉니다.
    if boxes_a.shape[-1] != 4 or boxes_b.shape[-1] != 4:  # 두 입력 모두 네 좌표를 갖는지 논리합 or로 확인합니다.
        raise ValueError("각 박스는 [x1, y1, x2, y2] 네 값이어야 합니다.")  # 계약 위반을 분명한 예외로 알립니다.

    intersection_top_left = np.maximum(boxes_a[..., :2], boxes_b[..., :2])  # 교집합의 시작점은 두 시작점 중 큰 값입니다.
    intersection_bottom_right = np.minimum(boxes_a[..., 2:], boxes_b[..., 2:])  # 교집합의 끝점은 두 끝점 중 작은 값입니다.
    intersection_size = np.maximum(intersection_bottom_right - intersection_top_left, 0.0)  # 겹치지 않을 때 음수가 되지 않게 0으로 자릅니다.
    intersection_area = intersection_size[..., 0] * intersection_size[..., 1]  # 교집합 폭과 높이를 곱합니다.

    union_area = box_area(boxes_a) + box_area(boxes_b) - intersection_area  # 두 넓이의 합에서 중복된 교집합을 한 번 뺍니다.
    return intersection_area / np.maximum(union_area, epsilon)  # 0으로 나누지 않도록 합집합에 작은 하한을 둡니다.


ground_truth = np.array([0.30, 0.30, 0.70, 0.70])  # 정답 박스를 모서리 좌표로 정의합니다.
prediction = np.array([0.40, 0.40, 0.80, 0.80])  # 오른쪽 아래로 조금 어긋난 예측 박스를 정의합니다.
iou = intersection_over_union(ground_truth, prediction)  # 두 박스의 겹침 정도를 계산합니다.
print(f"IoU: {iou:.4f}")  # f-string의 :.4f는 실수를 소수점 넷째 자리까지 출력합니다.

assert np.isclose(iou, 0.09 / 0.23)  # 교집합 0.09와 합집합 0.23으로 손계산한 값과 일치하는지 확인합니다.
assert intersection_over_union(ground_truth, ground_truth) == 1.0  # 같은 박스끼리의 IoU는 정확히 1이어야 합니다.
assert intersection_over_union(ground_truth, [0.0, 0.0, 0.1, 0.1]) == 0.0  # 겹치지 않는 박스의 IoU는 0이어야 합니다.

## 3. YOLO v1의 신뢰도와 클래스별 점수

논문의 신뢰도 정의는 `Pr(Object) × IoU`입니다. 추론 시 클래스 조건부 확률 `Pr(Classᵢ | Object)`을 곱하면 `Pr(Classᵢ) × IoU`가 되어 클래스마다 정렬 가능한 점수를 얻습니다.

In [ ]:
object_probability = 1.0  # 객체가 들어 있는 학습 셀을 가정하므로 객체 존재 확률을 1로 둡니다.
predicted_iou = float(iou)  # NumPy 스칼라를 일반 Python float로 바꿔 출력과 직렬화를 쉽게 합니다.
confidence = object_probability * predicted_iou  # YOLO v1 정의대로 객체 존재 확률과 IoU를 곱합니다.

conditional_class_probabilities = np.array([0.10, 0.70, 0.20])  # 객체가 있다는 조건에서 세 클래스의 확률을 정의합니다.
class_specific_scores = conditional_class_probabilities * confidence  # 브로드캐스팅으로 각 클래스 확률에 같은 신뢰도를 곱합니다.
best_class_index = int(np.argmax(class_specific_scores))  # argmax로 가장 큰 점수의 위치를 찾고 Python 정수로 바꿉니다.

print(f"confidence = {confidence:.4f}")  # 박스 자체의 신뢰도를 출력합니다.
print(f"class scores = {class_specific_scores}")  # 세 클래스의 최종 점수를 배열로 확인합니다.
print(f"best class index = {best_class_index}")  # 0부터 시작하는 최우수 클래스 인덱스를 출력합니다.

assert np.isclose(conditional_class_probabilities.sum(), 1.0)  # 예제의 조건부 클래스 확률 합이 1인지 확인합니다.
assert best_class_index == 1  # 두 번째 클래스의 확률이 가장 크므로 인덱스 1이어야 합니다.

## 직접 해볼 과제

1. 예측 박스를 정답 박스에서 멀리 옮기고 IoU와 클래스 점수의 변화를 관찰하세요.
2. 폭이나 높이가 0인 박스를 넣어 0으로 나누는 문제가 없는지 확인하세요.
3. 픽셀 좌표를 이미지 폭과 높이로 나눠 정규화한 뒤 같은 함수를 적용해 보세요.

다음 노트북에서는 이 기초 연산을 `7×7` 그리드 인코딩과 클래스별 NMS로 확장합니다.